# Auto MPG Prediction — Linear Regression

This project uses Linear Regression to predict a car's fuel efficiency (MPG) from its features.

The project focuses on exploring and preparing the dataset, training different Linear Regression models, and comparing their performance.

In [ ]:
# Used for numerical operations with arrays.
import numpy as np

# Used for loading, organizing and working with tabular data.
import pandas as pd

# Used to create and train the Linear Regression model.
from sklearn.linear_model import LinearRegression 

### 1. Load the Dataset

   The dataset is loaded into a pandas DataFrame so that it can be explored and prepared before training the model.

In [ ]:
df = pd.read_csv('data/auto-mpg.csv')

In [ ]:
df.head(5)

,mpg,cylinders,displacement,horsepower,weight,acceleration,model year,origin,car name
0,18.0,8,307.0,130,3504,12.0,70,1,chevrolet chevelle malibu
1,15.0,8,350.0,165,3693,11.5,70,1,buick skylark 320
2,18.0,8,318.0,150,3436,11.0,70,1,plymouth satellite
3,16.0,8,304.0,150,3433,12.0,70,1,amc rebel sst
4,17.0,8,302.0,140,3449,10.5,70,1,ford torino


### 2. Explore the Dataset
  

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           398 non-null    float64
 1   cylinders     398 non-null    int64  
 2   displacement  398 non-null    float64
 3   horsepower    398 non-null    object 
 4   weight        398 non-null    int64  
 5   acceleration  398 non-null    float64
 6   model year    398 non-null    int64  
 7   origin        398 non-null    int64  
 8   car name      398 non-null    object 
dtypes: float64(3), int64(4), object(2)
memory usage: 28.1+ KB


### Investigating the Horsepower Column

The `horsepower` column is stored as an object rather than a numerical data type, so its values are investigated before preparing the column.

In [ ]:
df['horsepower'].unique()

array(['130', '165', '150', '140', '198', '220', '215', '225', '190',
       '170', '160', '95', '97', '85', '88', '46', '87', '90', '113',
       '200', '210', '193', '?', '100', '105', '175', '153', '180', '110',
       '72', '86', '70', '76', '65', '69', '60', '80', '54', '208', '155',
       '112', '92', '145', '137', '158', '167', '94', '107', '230', '49',
       '75', '91', '122', '67', '83', '78', '52', '61', '93', '148',
       '129', '96', '71', '98', '115', '53', '81', '79', '120', '152',
       '102', '108', '68', '58', '149', '89', '63', '48', '66', '139',
       '103', '125', '133', '138', '135', '142', '77', '62', '132', '84',
       '64', '74', '116', '82'], dtype=object)

In [ ]:
(df['horsepower'] == '?').sum()

np.int64(6)

In [ ]:
df[df['horsepower'] == '?']

,mpg,cylinders,displacement,horsepower,weight,acceleration,model year,origin,car name
32,25.0,4,98.0,?,2046,19.0,71,1,ford pinto
126,21.0,6,200.0,?,2875,17.0,74,1,ford maverick
330,40.9,4,85.0,?,1835,17.3,80,2,renault lecar deluxe
336,23.6,4,140.0,?,2905,14.3,80,1,ford mustang cobra
354,34.5,4,100.0,?,2320,15.8,81,2,renault 18i
374,23.0,4,151.0,?,3035,20.5,82,1,amc concord dl


The investigation shows that some horsepower values are represented by `?` instead of numerical values. These values need to be handled before `horsepower` can be used as a numerical feature.

### 3. Prepare the Data
The `?` values in `horsepower` are replaced with missing values so that the column can be converted to a numerical data type.

In [ ]:
df['horsepower'] = df['horsepower'].replace('?', np.nan)

In [ ]:
df['horsepower'].isnull().sum()

np.int64(6)

In [ ]:
df['horsepower'] = pd.to_numeric(df['horsepower'])

The missing horsepower values are filled with the median horsepower before training the model.

In [ ]:
median_horsepower=df['horsepower'].median()
median_horsepower

93.0

In [ ]:
df['horsepower']=df['horsepower'].fillna(median_horsepower)
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           398 non-null    float64
 1   cylinders     398 non-null    int64  
 2   displacement  398 non-null    float64
 3   horsepower    398 non-null    float64
 4   weight        398 non-null    int64  
 5   acceleration  398 non-null    float64
 6   model year    398 non-null    int64  
 7   origin        398 non-null    int64  
 8   car name      398 non-null    object 
dtypes: float64(4), int64(4), object(1)
memory usage: 28.1+ KB


### 4. Explore Feature Relationships

Correlation is used to examine the linear relationships between the numerical variables in the dataset.

In [ ]:
df.corr(numeric_only=True)

,mpg,cylinders,displacement,horsepower,weight,acceleration,model year,origin
mpg,1.000000,-0.775396,-0.804203,-0.773522,-0.831741,0.420289,0.579267,0.563450
cylinders,-0.775396,1.000000,0.950721,0.841367,0.896017,-0.505419,-0.348746,-0.562543
displacement,-0.804203,0.950721,1.000000,0.895849,0.932824,-0.543684,-0.370164,-0.609409
horsepower,-0.773522,0.841367,0.895849,1.000000,0.862502,-0.686676,-0.413816,-0.452011
weight,-0.831741,0.896017,0.932824,0.862502,1.000000,-0.417457,-0.306564,-0.581024
acceleration,0.420289,-0.505419,-0.543684,-0.686676,-0.417457,1.000000,0.288137,0.205873
model year,0.579267,-0.348746,-0.370164,-0.413816,-0.306564,0.288137,1.000000,0.180662
origin,0.563450,-0.562543,-0.609409,-0.452011,-0.581024,0.205873,0.180662,1.000000


### 5. Model 1 — Initial Model

The initial model uses displacement, weight, horsepower, and cylinders as input features to predict MPG.

In [ ]:
X=df[['displacement','weight','horsepower','cylinders']]
y=df['mpg']

# Used to split the dataset into training and test sets.
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
model=LinearRegression()
model.fit(X_train,y_train)

LinearRegression()

In [ ]:
y_pred=model.predict(X_test)

The predictions are compared with the actual MPG values from the test set.

In [ ]:
comparison = pd.DataFrame({'Actual': y_test,'Predicted': y_pred})
comparison

,Actual,Predicted
198,33.0,32.703348
396,28.0,27.079260
33,19.0,25.184853
208,13.0,15.370495
93,14.0,13.791518
...,...,...
249,19.9,20.366164
225,17.5,19.970226
367,28.0,26.867236
175,29.0,31.268075


In [ ]:
# Used to calculate the Root Mean Squared Error (RMSE).
from sklearn.metrics import root_mean_squared_error

rmse = root_mean_squared_error(y_test, y_pred)

print("Root Mean Squared Error:", rmse)


Root Mean Squared Error: 3.830402800875051


### 6. Model 2 — Single-Feature Experiment

To test whether one feature could perform well on its own, a second model is trained using `weight` as the only input feature.

In [ ]:
X2=df[['weight']]
y2=df['mpg']

X_train2,X_test2,y_train2,y_test2=train_test_split(X2,y2,test_size=0.2,random_state=42)

In [ ]:
model2=LinearRegression()
model2.fit(X_train2,y_train2)

LinearRegression()

In [ ]:
y_pred2=model2.predict(X_test2)

In [ ]:
comparison2 = pd.DataFrame({'Actual': y_test2,'Predicted': y_pred2})
comparison2

,Actual,Predicted
198,33.0,32.771653
396,28.0,26.293302
33,19.0,26.223055
208,13.0,16.029409
93,14.0,13.711252
...,...,...
249,19.9,20.517423
225,17.5,19.307610
367,28.0,26.449407
175,29.0,31.663309


In [ ]:
rmse2 = root_mean_squared_error(y_test2, y_pred2)

print("Root Mean Squared Error:", rmse2)

Root Mean Squared Error: 3.8593860994510765


The single-feature model performed worse than the initial model. This suggests that the additional features in the initial model provided useful information for predicting MPG.

### 7. Model 3 — Adding Model Year

The third model adds `model year` to the features used in the initial model.

The purpose of this experiment is to test whether including model year improves prediction performance.

In [ ]:
X3=df[['displacement','weight','horsepower','cylinders','model year']]
y3=df['mpg']

X_train3,X_test3,y_train3,y_test3=train_test_split(X3,y3,test_size=0.2,random_state=42)

In [ ]:
model3=LinearRegression()
model3.fit(X_train3,y_train3)

LinearRegression()

In [ ]:
y_pred3=model3.predict(X_test3)

In [ ]:
comparison3 = pd.DataFrame({'Actual': y_test3,'Predicted': y_pred3})
comparison3

,Actual,Predicted
198,33.0,31.524712
396,28.0,30.596357
33,19.0,21.894229
208,13.0,17.004748
93,14.0,12.596808
...,...,...
249,19.9,22.612079
225,17.5,20.641181
367,28.0,30.687728
175,29.0,29.692901


In [ ]:
rmse3 = root_mean_squared_error(y_test3, y_pred3)

print("Root Mean Squared Error:", rmse3)

Root Mean Squared Error: 3.079828755629626


Adding `model year` improved the test RMSE compared with the initial model.



### 8. Final Model — Prediction Comparison

The final model's predictions are compared with the actual MPG values.

L1 Loss shows the absolute difference between each prediction and its actual value. A smaller L1 Loss means the individual prediction was closer to the actual MPG.

In [ ]:
comparison3 = pd.DataFrame({'Actual MPG': y_test3,'Predicted MPG': y_pred3,'L1 Loss': np.abs(y_pred3 - y_test3)})
comparison3


,Actual MPG,Predicted MPG,L1 Loss
198,33.0,31.524712,1.475288
396,28.0,30.596357,2.596357
33,19.0,21.894229,2.894229
208,13.0,17.004748,4.004748
93,14.0,12.596808,1.403192
...,...,...,...
249,19.9,22.612079,2.712079
225,17.5,20.641181,3.141181
367,28.0,30.687728,2.687728
175,29.0,29.692901,0.692901


### 9. Conclusion

Using Model 3 looks appropriate for this project because adding `model year` improved the test RMSE compared with the initial model.